In [ ]:
import os
import gradio as gr
from pdf_export import guardar_conversacion_pdf_mejorado
from utils_transcripcion import obtener_o_transcribir_audio, extraer_audio_con_ffmpeg
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import datetime
import hashlib
import openai
import pdfplumber
import docx
import pandas as pd

transcripciones_cache = {}
historial = []

c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extraer_texto_pdf(ruta_pdf):
    texto = ""
    with pdfplumber.open(ruta_pdf) as pdf:
        for pagina in pdf.pages:
            texto += pagina.extract_text() or ""
    return texto

def extraer_texto_docx(ruta_docx):
    doc = docx.Document(ruta_docx)
    return "\n".join([p.text for p in doc.paragraphs])

def extraer_texto_excel(ruta_excel):
    texto = ""
    xls = pd.ExcelFile(ruta_excel)
    for nombre_hoja in xls.sheet_names:
        df = pd.read_excel(xls, nombre_hoja)
        texto += df.to_string(index=False)
    return texto

In [3]:
import os

def conversar(archivo, mensaje_usuario, chat_historial):
    if archivo is None:
        return chat_historial, "⚠️ Por favor sube un archivo primero."

    # Detecta extensión correctamente si archivo es file-like o string
    ext = os.path.splitext(archivo.name)[1].lower() if hasattr(archivo, 'name') else os.path.splitext(archivo)[1].lower()
    texto_documento = None

    if ext in [".mp3", ".wav", ".m4a"]:
        archivo_audio = archivo
        transcripcion = obtener_o_transcribir_audio(archivo_audio)
        texto_documento = transcripcion
    elif ext in [".mp4", ".mov", ".avi", ".mkv"]:
        archivo_audio = extraer_audio_con_ffmpeg(archivo)
        if archivo_audio is None:
            return chat_historial, "No se pudo extraer el audio del video."
        transcripcion = obtener_o_transcribir_audio(archivo_audio)
        texto_documento = transcripcion
    elif ext == ".pdf":
        texto_documento = extraer_texto_pdf(archivo)
    elif ext == ".docx":
        texto_documento = extraer_texto_docx(archivo)
    elif ext in [".xlsx", ".xls"]:
        texto_documento = extraer_texto_excel(archivo)
    else:
        return chat_historial, f" Formato no soportado: {ext}"

    if not texto_documento:
        return chat_historial, "No se pudo extraer texto del archivo."

    mensajes = [{"role": "system", "content": "Eres un asistente educativo que responde en formato Markdown. Puedes analizar el contenido de audio o texto de archivos subidos."}]
    for user_msg, assistant_msg in chat_historial:
        mensajes.append({"role": "user", "content": user_msg})
        mensajes.append({"role": "assistant", "content": assistant_msg})

    if not chat_historial:
        mensaje_completo = f"{mensaje_usuario}\n\nCONTENIDO DEL ARCHIVO:\n{texto_documento}"
    else:
        mensaje_completo = mensaje_usuario

    mensajes.append({"role": "user", "content": mensaje_completo})

    respuesta = openai.chat.completions.create(
        model="gpt-4o",
        messages=mensajes,
        max_tokens=700
    ).choices[0].message.content

    chat_historial.append((mensaje_usuario, respuesta))
    return chat_historial, "✅ Respondido"

In [4]:
# Importa la función mejorada para exportar PDF
from pdf_export import guardar_conversacion_pdf_mejorado

def exportar_pdf_mejorado(chat_historial):

    try:
        archivo_pdf = guardar_conversacion_pdf_mejorado(chat_historial)
        print(f" PDF generado exitosamente: {archivo_pdf}")
        return archivo_pdf  # Gradio File component en versiones recientes
    except Exception as e:
        print(f" Error al generar PDF: {str(e)}")
        return None


In [5]:
def extraer_audio_con_ffmpeg(video_path, output_audio_path="audio_extraido.wav"):
    try:
        (
            ffmpeg
            .input(video_path)
            .output(output_audio_path, format='wav', acodec='pcm_s16le', ac=1, ar='16000')
            .overwrite_output()
            .run(quiet=True)
        )
        return output_audio_path
    except ffmpeg.Error as e:
        return None




def responder_con_gpt4(pregunta, contexto):
    prompt = f"""
Eres un asistente educativo que responde en formato Markdown. Usa listas, bloques de código y **fórmulas matemáticas con doble signo de dólar `$$`** para que se rendericen correctamente.

--- CONTEXTO TRANSCRITO ---
{contexto}

--- PREGUNTA DEL USUARIO ---
{pregunta}

Responde usando **Markdown** con formato matemático en bloque usando `$$`.
"""
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800
    )
    return response.choices[0].message.content


In [6]:
def interfaz_gradio(archivo, pregunta_usuario):
    if archivo is None:
        return "No se proporcionó ningún archivo.", ""
    
    ext = os.path.splitext(archivo)[1].lower()

    if ext in [".mp3", ".wav", ".m4a"]:
        archivo_audio = archivo
    elif ext in [".mp4", ".mov", ".avi", ".mkv"]:
        archivo_audio = extraer_audio_con_ffmpeg(archivo)
        if archivo_audio is None:
            return "No se pudo extraer el audio del video.", ""
    else:
        return f"Tipo de archivo no soportado: {ext}", ""

    transcripcion = transcribir_audio(archivo_audio)

    if not pregunta_usuario.strip():
        return transcripcion, "💡 Escribe una pregunta para recibir una respuesta."

    respuesta = responder_con_gpt4(pregunta_usuario, transcripcion)
    
    return transcripcion, respuesta


In [ ]:
def escuchar_y_transcribir(audio):
    if audio is None:
        return "", "No se grabó audio."
    transcripcion = obtener_o_transcribir_audio(audio)
    return transcripcion, transcripcion  # actualiza tanto el estado como el textbox

def preguntar_sobre_transcripcion(transcripcion, pregunta, chat_historial):
    if not transcripcion or transcripcion.strip() == "":
        return chat_historial, "No hay transcripción disponible."
    mensajes = [{"role": "system", "content": "Eres un asistente educativo que responde en formato Markdown. Puedes analizar el contenido de audio transcrito."}]
    for user_msg, assistant_msg in chat_historial:
        mensajes.append({"role": "user", "content": user_msg})
        mensajes.append({"role": "assistant", "content": assistant_msg})

    if not chat_historial:
        mensaje_completo = f"{pregunta}\n\nCONTENIDO DE LA CLASE TRANSCRITO:\n{transcripcion}"
    else:
        mensaje_completo = pregunta

    mensajes.append({"role": "user", "content": mensaje_completo})

    respuesta = openai.chat.completions.create(
        model="gpt-4o",
        messages=mensajes,
        max_tokens=700
    ).choices[0].message.content

    chat_historial.append((pregunta, respuesta))
    return chat_historial, respuesta

with gr.Blocks(title="Asistente Multimodal con GPT-4o") as demo:
    gr.Markdown("## 🎓 Asistente Multimodal con GPT-4o")
    gr.Markdown("Sube un audio, video, documento, o graba con el micrófono. Pregunta lo que quieras y GPT-4o te responde con soporte para código, listas y fórmulas matemáticas en Markdown.")

    with gr.Tab("Archivo"):
        archivo = gr.File(
            label="Sube tu archivo",
            file_types=[".pdf", ".docx", ".xlsx", ".xls", ".mp3", ".wav", ".m4a", ".mp4", ".mov", ".avi", ".mkv"]
        )
        chatbot = gr.Chatbot(label="🗨️ Conversación")
        pregunta = gr.Textbox(label="💬 Escribe tu pregunta", placeholder="Ej. ¿Qué se dijo en la explicación?", lines=1)
        estado = gr.Markdown("🟡 Esperando pregunta...")

        enviar = gr.Button("🚀 Enviar")
        limpiar = gr.Button("🧹 Limpiar conversación")
        btn_pdf = gr.Button("📄 Guardar conversación como PDF")
        archivo_pdf_gr = gr.File(label="📄 Descarga tu conversación")

        # Funciones conectadas
        def limpiar_historial():
            return [], "🟡 Conversación reiniciada."

        def exportar_pdf(chat_historial):
            archivo_pdf = guardar_conversacion_pdf(chat_historial)
            return gr.File.update(value=archivo_pdf)

        limpiar.click(fn=limpiar_historial, outputs=[chatbot, estado])
        btn_pdf.click(fn=exportar_pdf_mejorado, inputs=[chatbot], outputs=[archivo_pdf_gr])
        enviar.click(fn=conversar, inputs=[archivo, pregunta, chatbot], outputs=[chatbot, estado])
        pregunta.submit(fn=conversar, inputs=[archivo, pregunta, chatbot], outputs=[chatbot, estado])

    with gr.Tab("Micrófono"):
        audio_input = gr.Audio(type="filepath", label="Graba la clase o pregunta")
        boton_transcribir = gr.Button("Transcribir")
        transcripcion_output = gr.Textbox(label="Transcripción")
        pregunta_mic = gr.Textbox(label="💬 Haz una pregunta sobre la transcripción", placeholder="Ej. ¿Qué dijo el profesor sobre la tarea?", lines=1)
        chatbot_mic = gr.Chatbot(label="🗨️ Conversación (micrófono)")
        estado_mic = gr.Markdown("🟡 Esperando grabación o pregunta...")
        transcripcion_state = gr.State("")  # Estado para la transcripción actual

        # 1. Al transcribir, actualiza tanto el estado como el textbox visible
        boton_transcribir.click(
            escuchar_y_transcribir,
            inputs=audio_input,
            outputs=[transcripcion_state, transcripcion_output]
        )
        # 2. Al preguntar, usa SIEMPRE el estado más reciente
        pregunta_mic.submit(
            preguntar_sobre_transcripcion,
            inputs=[transcripcion_state, pregunta_mic, chatbot_mic],
            outputs=[chatbot_mic, estado_mic]
        )

demo.launch()

C:\Users\Arria\AppData\Local\Temp\ipykernel_6076\2133118501.py:42: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="🗨️ Conversación")
C:\Users\Arria\AppData\Local\Temp\ipykernel_6076\2133118501.py:69: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_mic = gr.Chatbot(label="🗨️ Conversación (micrófono)")


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\starlette\applications.py", line 112, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\Arria\anaconda3\envs\libretainteligente\Lib\site-packages\starlett